# EU AI Act — LLM-Based Article → Recital Map

The naive 3-signal approach in `eu_ai_act_retrieval.ipynb` builds the map using:
- explicit article references in recital metadata (only catches ~14/113 articles)
- keyword overlap between recital text and article *title* (too sparse, wrong matches)
- chapter-position fallback (pure positional, no semantic reasoning)

That's unreliable. EU legislative drafting is intentional — recitals explain the *why* behind
articles using rationale language that rarely shares vocabulary with the article text itself.
The only thing that can reliably judge "does recital R explain the purpose of article A?" is
something that understands legal intent — an LLM.

**What we build here, step by step:**
```
1.  Install & imports
2.  Load chunks → reconstruct full article texts (merge multi-chunk articles)
3.  Understand the problem: why naive fails on a concrete example
4.  Embed recitals and articles (qwen3-embedding-8b via OpenRouter)
5.  Candidate pre-filter with guaranteed layer for explicit refs
6.  K-sweep: pick TOP_K_CANDIDATES empirically
7.  LLM prompt design
8.  Single article test — Article 5 (gold standard)
9.  Test on Articles 16 and 53
10. Batch all 113 articles with retry + resume
11. Build final map and compare vs naive
12. Quality audit — spot-check 10 articles
13. Strict second pass for uncovered articles
14. Save
```

**Stack:**
- `langchain_openai.ChatOpenAI` → `stepfun/step-3.5-flash:free` via OpenRouter
- `langchain_openai.OpenAIEmbeddings` → `qwen/qwen3-embedding-8b` via OpenRouter
- API key from `.env` via `python-dotenv`


## 0. Install

In [ ]:
import subprocess, sys

pkgs = ["langchain-openai", "python-dotenv", "numpy", "tqdm"]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q", "--break-system-packages"],
        capture_output=True, text=True
    )
    print(f"  {'✓' if r.returncode==0 else '✗'}  {pkg}")

print("\nAll packages ready")

## 1. Imports & Config

In [3]:
import json, os, re, time, textwrap
from pathlib     import Path
from collections import defaultdict

import numpy as np
from tqdm  import tqdm
from dotenv import load_dotenv

from langchain_openai       import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR       = Path("./chunker_output")   # folder containing all_chunks.json
OUTPUT_MAP     = DATA_DIR / "article_recital_map_llm.json"
OUTPUT_REASONS = DATA_DIR / "article_recital_reasons_llm.json"
INTERMEDIATE   = DATA_DIR / "_arm_intermediate.json"
OLD_MAP        = DATA_DIR / "article_recital_map.json"   # naive map for comparison

# ── OpenRouter ────────────────────────────────────────────────────────────────
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
OPENROUTER_KEY  = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not found.\n"
        "Add it to your .env file: OPENROUTER_API_KEY=sk-or-..."
    )

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model       = "stepfun/step-3.5-flash:free",
    api_key     = OPENROUTER_KEY,
    base_url    = OPENROUTER_BASE,
    max_retries = 2,
    temperature = 0.0,   # deterministic — same map every run
)

# ── Embeddings ────────────────────────────────────────────────────────────────
embedding_model = OpenAIEmbeddings(
    model    = "qwen/qwen3-embedding-8b",
    base_url = OPENROUTER_BASE,
    api_key  = OPENROUTER_KEY,
)

# ── Tunable params ────────────────────────────────────────────────────────────
TOP_K_CANDIDATES = 30   # candidate recitals per article sent to LLM
                        # will tune this in cell 6 using the K-sweep
EMBED_BATCH_SIZE = 16   # texts per embedding API call
RETRY_ATTEMPTS   = 3
RETRY_DELAY      = 3.0
REQUEST_DELAY    = 0.5  # courtesy delay between article LLM calls

print(f"LLM model       : {llm.model_name}")
print(f"Embedding model : {embedding_model.model}")
print(f"TOP_K_CANDIDATES: {TOP_K_CANDIDATES}")
print(f"Output map      : {OUTPUT_MAP}")

LLM model       : stepfun/step-3.5-flash:free
Embedding model : qwen/qwen3-embedding-8b
TOP_K_CANDIDATES: 30
Output map      : chunker_output/article_recital_map_llm.json


---
## 2. Load Chunks & Reconstruct Full Article Texts

Articles that are long were split into multiple chunks (chunk_index 0, 1, 2, …).  
For the LLM to judge whether a recital explains an article, it needs the **full article text**,
not just one chunk. We group by `article_num` and concatenate in chunk_index order.

In [4]:
with open(DATA_DIR / "all_chunks.json") as f:
    raw = json.load(f)

recital_chunks = [c for c in raw if c["metadata"]["content_type"] == "recital"]
article_chunks = [c for c in raw if c["metadata"]["content_type"] == "article"]
annex_chunks   = [c for c in raw if c["metadata"]["content_type"] == "annex"]

print(f"Total chunks : {len(raw)}")
print(f"  recitals   : {len(recital_chunks)}")
print(f"  articles   : {len(article_chunks)}  (post-split — some articles span multiple chunks)")
print(f"  annexes    : {len(annex_chunks)}")

Total chunks : 452
  recitals   : 180
  articles   : 260  (post-split — some articles span multiple chunks)
  annexes    : 12


In [5]:
# Group article chunks by article_num and merge in chunk_index order
art_chunk_groups = defaultdict(list)
for c in article_chunks:
    m = c["metadata"]["article"]
    art_chunk_groups[m["article_num"]].append((m["chunk_index"], c))

full_articles = {}   # article_num → {article_title, full_text, num_chunks, ...}
for num, clist in art_chunk_groups.items():
    clist.sort(key=lambda x: x[0])
    merged = "\n\n".join(c["page_content"] for _, c in clist)
    meta0  = clist[0][1]["metadata"]["article"]
    full_articles[num] = {
        "article_num"   : num,
        "article_title" : meta0["article_title"],
        "chapter_num"   : meta0.get("chapter_num", ""),
        "chapter_title" : meta0.get("chapter_title", ""),
        "section_num"   : meta0.get("section_num", ""),
        "section_title" : meta0.get("section_title", ""),
        "full_text"     : merged,
        "num_chunks"    : len(clist),
        "char_len"      : len(merged),
    }

article_nums_sorted = sorted(full_articles.keys())
known_article_nums  = set(full_articles.keys())

print(f"Reconstructed {len(full_articles)} unique articles")
print()

multi = sorted(
    [(n, v) for n, v in full_articles.items() if v["num_chunks"] > 1],
    key=lambda x: x[1]["num_chunks"], reverse=True
)
print(f"Articles spanning MULTIPLE chunks — {len(multi)} total")
print(f"  {'Art':>4}  {'Title':<48}  {'Chunks':>6}  {'Chars':>7}")
print("  " + "-" * 72)
for n, v in multi[:12]:
    print(f"  {n:4d}  {v['article_title'][:48]:<48}  {v['num_chunks']:6d}  {v['char_len']:7d}")

print()
print(f"  This is why we reconstruct — the LLM needs the whole text, not one slice of it.")

Reconstructed 113 unique articles

Articles spanning MULTIPLE chunks — 71 total
   Art  Title                                             Chunks    Chars
  ------------------------------------------------------------------------
     3  Definitions                                           13    17793
     5  Prohibited AI practices                                8    11349
    57  AI regulatory sandboxes                                7     9409
    26  Obligations of deployers of high-risk AI systems       6     7708
    60  Testing of high-risk AI systems in real world co       6     7916
    74  Market surveillance and control of AI systems in       6     7325
    36  Changes to notifications                               5     6526
     2  Scope                                                  4     4750
    10  Data and data governance                               4     4533
    43  Conformity assessment                                  4     4921
    50  Transparency obligation

In [6]:
# Recital index
recital_index = {c["metadata"]["recital"]["recital_num"]: c for c in recital_chunks}
recital_nums_sorted = sorted(recital_index.keys())
print(f"Recital index: {len(recital_index)} recitals  "
      f"({min(recital_nums_sorted)} → {max(recital_nums_sorted)})")
print()

# ── Build explicit_art_to_recitals with phantom-ref guard ───────────────────
#
# The chunker's _article_refs() regex picks up ANY "Article N" in recital text,
# including cross-references to OTHER EU regulations
# (e.g. "Article 261 of Regulation (EU) 2019/...").
# Those phantom numbers are not in full_articles and would crash get_candidate_recitals.
# Fix: only keep article numbers that are in known_article_nums.

_raw_explicit = defaultdict(set)
for c in recital_chunks:
    rn   = c["metadata"]["recital"]["recital_num"]
    refs = c["metadata"]["recital"]["referenced_articles"]
    for ref in refs:
        try:
            _raw_explicit[int(ref.split()[1])].add(rn)
        except:
            pass

explicit_art_to_recitals = {
    k: v for k, v in _raw_explicit.items() if k in known_article_nums
}

phantom_nums = sorted(set(_raw_explicit) - known_article_nums)
print(f"Raw article refs found in recital metadata:")
print(f"  All nums  : {sorted(_raw_explicit.keys())}")
print(f"  Phantoms  : {phantom_nums}  ← from other EU regulations, filtered out")
print()
print(f"Valid explicit links ({len(explicit_art_to_recitals)} articles):")
for art_num, rnums in sorted(explicit_art_to_recitals.items()):
    title = full_articles[art_num]["article_title"]
    print(f"  Art {art_num:3d} ({title[:40]:<40})  ← recitals {sorted(rnums)}")

Recital index: 180 recitals  (1 → 180)

Raw article refs found in recital metadata:
  All nums  : [1, 2, 3, 4, 5, 6, 10, 13, 16, 18, 24, 42, 54, 261]
  Phantoms  : [261]  ← from other EU regulations, filtered out

Valid explicit links (13 articles):
  Art   1 (Subject matter`                         )  ← recitals [180]
  Art   2 (Scope                                   )  ← recitals [6, 121]
  Art   3 (Definitions                             )  ← recitals [53]
  Art   4 (AI literacy                             )  ← recitals [24, 53, 94, 104, 106, 140]
  Art   5 (Prohibited AI practices                 )  ← recitals [40, 41, 176]
  Art   6 (Classification rules for high-risk AI sy)  ← recitals [6]
  Art  10 (Data and data governance                )  ← recitals [38, 39, 94, 140]
  Art  13 (Transparency and provision of informatio)  ← recitals [93]
  Art  16 (Obligations of providers of high-risk AI)  ← recitals [4, 40, 136]
  Art  18 (Documentation keeping                   )  ← recital

---
## 3. Why the Naive Keyword Signal Fails

Concretely: read what naive assigned to Article 16 and ask whether those recitals
genuinely explain its provider obligations, or just share EU boilerplate.

In [7]:
naive_map = {}
if OLD_MAP.exists():
    with open(OLD_MAP) as f:
        naive_map = {int(k): v for k, v in json.load(f).items()}
    naive_total   = sum(len(v) for v in naive_map.values())
    naive_covered = sum(1 for v in naive_map.values() if v)
    print(f"Naive map loaded")
    print(f"  Articles with ≥1 recital : {naive_covered} / {len(full_articles)}")
    print(f"  Total assignments        : {naive_total}")
    print(f"  Avg recitals/article     : {naive_total/max(naive_covered,1):.1f}")
else:
    print("No naive map found. Run eu_ai_act_retrieval.ipynb first for the comparison.")

Naive map loaded
  Articles with ≥1 recital : 108 / 113
  Total assignments        : 1428
  Avg recitals/article     : 13.2


In [8]:
INSPECT_NUM = 16
art16 = full_articles[INSPECT_NUM]

print(f"=== Article {INSPECT_NUM}: {art16['article_title']} ===")
print(f"  Chapter : {art16['chapter_num']} — {art16['chapter_title']}")
print(f"  Chars   : {art16['char_len']}   Chunks: {art16['num_chunks']}")
print()
print("Article text (first 400 chars):")
print(art16["full_text"][:400])
print()

if naive_map:
    naive_16 = naive_map.get(INSPECT_NUM, [])
    print(f"Naive assigned {len(naive_16)} recitals: {naive_16[:12]}")
    print()
    print("First 5 naive recitals — do they explain Art 16's provider obligations?")
    for rn in naive_16[:5]:
        if rn in recital_index:
            text = recital_index[rn]["page_content"][:230].replace("\n", " ")
            print(f"  Recital ({rn:3d}): {text}…")
    print()
    print("^ Many of those just share EU regulatory boilerplate.")
    print("  That's the false-positive problem keyword overlap creates.")

=== Article 16: Obligations of providers of high-risk AI systems ===
  Chapter : CHAPTER III — CHAPTERIII
  Chars   : 1680   Chunks: 2

Article text (first 400 chars):
Providers of high-risk AI systems shall:
(a) ensure that their high-risk AI systems are compliant with the requirements set out in Section 2;
(b) indicateonthehigh-riskAIsystemor,wherethatisnotpossible,onitspackagingoritsaccompanyingdocumentation,
as applicable, their name, registered trade name or registered trade mark,the address at whichtheycanbe contacted;
(c) have a quality management system 

Naive assigned 95 recitals: [3, 4, 7, 9, 10, 14, 20, 21, 22, 26, 40, 46]

First 5 naive recitals — do they explain Art 16's provider obligations?
  Recital (  3): AIsystemscanbeeasilydeployedinalargevarietyofsectorsoftheeconomyandmanypartsofsociety,including across borders, and can easily circulate throughout the Union. Certain Member States have already explored the adoptionof national rul…
  Recital (  4): Positionof theEuro

---
## 4. Embed Recitals and Articles

**`qwen/qwen3-embedding-8b`** — 8B param, 4096-dim output, trained on multilingual formal text.
Handles the rationale↔obligation language gap that breaks smaller general-purpose models.

Called via `langchain_openai.OpenAIEmbeddings` pointed at OpenRouter.

In [9]:
def embed_batch(texts: list, delay: float = 0.2) -> np.ndarray:
    """
    Embed texts in EMBED_BATCH_SIZE batches.
    Returns float32 array (N, dim), L2-normalised (dot product = cosine similarity).
    """
    all_vecs  = []
    n_batches = (len(texts) + EMBED_BATCH_SIZE - 1) // EMBED_BATCH_SIZE

    for i in range(n_batches):
        start = i * EMBED_BATCH_SIZE
        end   = min(start + EMBED_BATCH_SIZE, len(texts))
        batch = texts[start:end]

        for attempt in range(RETRY_ATTEMPTS):
            try:
                vecs = embedding_model.embed_documents(batch)   # list[list[float]]
                all_vecs.extend(vecs)
                break
            except Exception as e:
                print(f"  Batch {i+1}/{n_batches} attempt {attempt+1} failed: {e}")
                if attempt < RETRY_ATTEMPTS - 1:
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"  ⚠  Inserting zero vectors for batch {i+1}")
                    all_vecs.extend([[0.0]] * len(batch))   # dim placeholder

        if (i + 1) % 5 == 0 or (i + 1) == n_batches:
            print(f"  Embedded {end}/{len(texts)} texts  ({i+1}/{n_batches} batches)")
        time.sleep(delay)

    matrix = np.array(all_vecs, dtype=np.float32)     # (N, dim)
    norms  = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms  = np.where(norms == 0, 1.0, norms)
    return matrix / norms


print("embed_batch() defined ✓")
print()

# Quick sanity check before the full run
sanity_texts = [
    "Providers of high-risk AI systems shall implement a risk management system throughout the lifecycle.",
    "Whereas it is necessary to impose risk management obligations on AI providers to ensure safety.",
    "The quick brown fox jumps over the lazy dog in the forest.",
]
sanity_embs = embed_batch(sanity_texts)
sim_legal   = float(sanity_embs[0] @ sanity_embs[1])
sim_random  = float(sanity_embs[0] @ sanity_embs[2])
print(f"Embedding quality sanity check:")
print(f"  sim(art-like, recital-like) = {sim_legal:.4f}  ← expect HIGH")
print(f"  sim(art-like, random text)  = {sim_random:.4f}  ← expect LOW")
print(f"  {'✓ OK' if sim_legal > sim_random else '✗ Model ranking looks wrong — check endpoint'}")

embed_batch() defined ✓

  Embedded 3/3 texts  (1/1 batches)
Embedding quality sanity check:
  sim(art-like, recital-like) = 0.4210  ← expect HIGH
  sim(art-like, random text)  = 0.3441  ← expect LOW
  ✓ OK


In [10]:
print(f"Embedding {len(recital_nums_sorted)} recitals …")
print()

recital_texts      = [recital_index[n]["page_content"] for n in recital_nums_sorted]
recital_embeddings = embed_batch(recital_texts, delay=0.15)

print()
print(f"Recital embeddings : {recital_embeddings.shape}")
print(f"  L2-normalised    : {np.allclose(np.linalg.norm(recital_embeddings, axis=1), 1.0, atol=1e-5)}")

Embedding 180 recitals …

  Embedded 80/180 texts  (5/12 batches)
  Embedded 160/180 texts  (10/12 batches)
  Embedded 180/180 texts  (12/12 batches)

Recital embeddings : (180, 4096)
  L2-normalised    : True


In [11]:
# Article embed text: title repeated 3× + chapter/section + full_text
#
# Why repeat the title?
# The title is the most discriminative signal. For long articles the model's
# effective attention decays over distance; prepending the title 3× ensures the
# embedding is anchored to what the article is ABOUT, not just its last paragraph.

def article_embed_text(art: dict) -> str:
    title   = art["article_title"]
    chapter = f"Chapter {art['chapter_num']}: {art['chapter_title']}"
    section = f"Section {art['section_num']}: {art['section_title']}" if art.get("section_title") else ""
    header  = f"{title}. {title}. {title}.\n{chapter}.\n{section}."
    return f"{header}\n\n{art['full_text']}"


print("Example embed text for Article 5 (first 350 chars):")
print("─" * 50)
print(article_embed_text(full_articles[5])[:350])
print("\n... [full article text continues] ...")
print()

print(f"Embedding {len(article_nums_sorted)} articles …")
print()

article_embed_texts = [article_embed_text(full_articles[n]) for n in article_nums_sorted]
article_embeddings  = embed_batch(article_embed_texts, delay=0.15)

print()
print(f"Article embeddings : {article_embeddings.shape}")
print(f"  L2-normalised    : {np.allclose(np.linalg.norm(article_embeddings, axis=1), 1.0, atol=1e-5)}")

Example embed text for Article 5 (first 350 chars):
──────────────────────────────────────────────────
Prohibited AI practices. Prohibited AI practices. Prohibited AI practices.
Chapter CHAPTER II: CHAPTERII.
.

1. The following AI practices shall be prohibited:
(a) theplacingonthemarket,theputtingintoserviceortheuseofanAIsystemthatdeployssubliminaltechniquesbeyond
a person’s consciousness or purposefully manipulative or deceptive techniques, with t

... [full article text continues] ...

Embedding 113 articles …

  Embedded 80/113 texts  (5/8 batches)
  Embedded 113/113 texts  (8/8 batches)

Article embeddings : (113, 4096)
  L2-normalised    : True


---
## 5. Candidate Pre-Filter with Guaranteed Layer

Two layers:
- **Guaranteed**: explicit metadata refs always included at score=2.0 (above any cosine sim)
- **Embedding**: top-K cosine similarity fills remaining slots

This makes recall on explicit refs **100% by construction** regardless of embedding quality.

In [12]:
# O(1) lookup: article_num → index in article_nums_sorted
art_num_to_idx = {n: i for i, n in enumerate(article_nums_sorted)}


def get_candidate_recitals(article_num: int, top_k: int = TOP_K_CANDIDATES) -> list:
    """
    Returns list of (recital_num, score) sorted descending.

    Layer 1 — Guaranteed: explicit metadata refs, score=2.0, always at top.
    Layer 2 — Embedding: cosine similarity, fills remaining slots.

    Returns [] for phantom article numbers (not in our embeddings).
    """
    if article_num not in art_num_to_idx:
        return []   # phantom ref from another regulation

    # Layer 1
    explicit_rnums = explicit_art_to_recitals.get(article_num, set())
    guaranteed     = [(rn, 2.0) for rn in sorted(explicit_rnums) if rn in recital_index]
    guaranteed_set = {rn for rn, _ in guaranteed}

    # Layer 2
    art_emb  = article_embeddings[art_num_to_idx[article_num]]
    sims     = recital_embeddings @ art_emb     # cosine sim (both normalised)
    n_fetch  = top_k + len(guaranteed)          # overfetch to cover dedup loss
    top_idxs = np.argsort(sims)[::-1][:n_fetch]

    embed_ranked = [
        (recital_nums_sorted[i], float(sims[i]))
        for i in top_idxs
        if recital_nums_sorted[i] not in guaranteed_set
    ]

    return (guaranteed + embed_ranked)[:top_k]


print("get_candidate_recitals() defined ✓")
print()

# Phantom guard test
for pn in phantom_nums[:3]:
    print(f"  get_candidate_recitals({pn}) → {get_candidate_recitals(pn)}  ← empty, no crash ✓")

print()

# Guaranteed layer test
print("Guaranteed layer verification:")
for art_num in [5, 16, 10, 4]:
    cands = get_candidate_recitals(art_num, top_k=TOP_K_CANDIDATES)
    cnums = {rn for rn, _ in cands}
    expl  = sorted(explicit_art_to_recitals.get(art_num, []))
    ok    = all(rn in cnums for rn in expl)
    title = full_articles[art_num]["article_title"]
    print(f"  {'✓' if ok else '✗'}  Art {art_num:3d} ({title[:38]:<38})  explicit={expl}")

get_candidate_recitals() defined ✓

  get_candidate_recitals(261) → []  ← empty, no crash ✓

Guaranteed layer verification:
  ✓  Art   5 (Prohibited AI practices               )  explicit=[40, 41, 176]
  ✓  Art  16 (Obligations of providers of high-risk )  explicit=[4, 40, 136]
  ✓  Art  10 (Data and data governance              )  explicit=[38, 39, 94, 140]
  ✓  Art   4 (AI literacy                           )  explicit=[24, 53, 94, 104, 106, 140]


---
## 6. K-Sweep — Picking TOP_K_CANDIDATES

The embedding-only recall tells us how many non-explicit genuine links the model can
find, and at what K. This is the empirical basis for the TOP_K_CANDIDATES choice.

Higher K = better discovery = longer/costlier prompts. Sweet spot is usually 30–50.

In [13]:
# Ground-truth pairs: (article_num, recital_num) from explicit metadata
explicit_pairs = [
    (a, rn)
    for a, rnums in explicit_art_to_recitals.items()
    if a in art_num_to_idx
    for rn in rnums
]

print(f"Ground-truth pairs for recall measurement: {len(explicit_pairs)}")
print()
print("Embedding-only recall (no guaranteed layer) at each K:")
print(f"  {'K':>4}  {'Recall %':>10}  {'Hits':>5}/{len(explicit_pairs):<5}  Bar")
print("  " + "-" * 55)

for K in [10, 20, 30, 40, 50, 60, 80, 100]:
    hits = sum(
        1 for art_num, rn in explicit_pairs
        if rn in {
            recital_nums_sorted[i]
            for i in np.argsort(recital_embeddings @ article_embeddings[art_num_to_idx[art_num]])[::-1][:K]
        }
    )
    pct  = 100 * hits / len(explicit_pairs)
    bar  = "█" * int(pct / 5)
    mark = " ← current" if K == TOP_K_CANDIDATES else ""
    print(f"  {K:4d}  {pct:9.1f}%  {hits:5d}/{len(explicit_pairs):<5}  {bar}{mark}")

print()
print("Note: even at low embedding recall, explicit refs are always in candidates")
print("because of the guaranteed layer. This sweep measures the quality of the")
print("embedding for discovering non-explicit (majority) genuine links.")

Ground-truth pairs for recall measurement: 27

Embedding-only recall (no guaranteed layer) at each K:
     K    Recall %   Hits/27     Bar
  -------------------------------------------------------
    10       11.1%      3/27     ██
    20       14.8%      4/27     ██
    30       14.8%      4/27     ██ ← current
    40       18.5%      5/27     ███
    50       25.9%      7/27     █████
    60       25.9%      7/27     █████
    80       44.4%     12/27     ████████
   100       63.0%     17/27     ████████████

Note: even at low embedding recall, explicit refs are always in candidates
because of the guaranteed layer. This sweep measures the quality of the
embedding for discovering non-explicit (majority) genuine links.


---
## 7. LLM Prompt Design

We ask the LLM to distinguish recitals that explain *why* an article exists
from those that merely share EU regulatory vocabulary.

Temperature 0 — deterministic. JSON output with a reason per link for auditability.

In [14]:
SYSTEM_PROMPT = textwrap.dedent("""
You are an expert on EU legislative drafting conventions, specialising in the EU AI Act (Regulation 2024/1689).

Your task: given one article from the EU AI Act and a list of candidate recitals,
identify which recitals provide the LEGISLATIVE RATIONALE or INTENT specifically for that article.

A recital IS relevant when it:
  - Explains WHY that article's obligation, prohibition, or requirement was introduced
  - Provides the policy justification or fundamental-rights basis for that article
  - Clarifies the scope or interpretation of that article's key concepts
  - Was placed by the EU legislator to contextualise that specific provision

A recital is NOT relevant when it:
  - Only shares common EU regulatory boilerplate ("placing on the market", "high-risk", "shall", etc.)
  - Is a general preamble applying equally to the whole regulation
  - Relates to a DIFFERENT article that uses similar language
  - Discusses the general risk-based approach without linking to this article's specific obligation

Be selective and precise. Most articles have 3–10 genuinely linked recitals.
Do NOT link a recital just because it mentions a concept that appears in many articles.

Respond ONLY with valid JSON — no preamble, no markdown fences, nothing outside the JSON:
{"article_num": <int>, "linked_recitals": [{"recital_num": <int>, "reason": "<one sentence>"}]}
""").strip()


def build_user_message(article_num: int, candidate_recital_nums: list) -> str:
    art = full_articles[article_num]

    # Truncate long articles — 3500 chars ≈ 875 tokens
    # Embeddings used full text, so truncation here doesn't affect candidate selection
    art_text = art["full_text"]
    if len(art_text) > 3500:
        art_text = art_text[:3500] + "\n[...truncated]"

    article_block = (
        f"ARTICLE {article_num}: {art['article_title']}\n"
        f"Chapter: {art['chapter_num']} — {art['chapter_title']}\n"
        f"Section: {art.get('section_num','')} {art.get('section_title','')}\n\n"
        f"{art_text}"
    )

    recital_lines = []
    for rn in candidate_recital_nums:
        if rn not in recital_index:
            continue
        text = recital_index[rn]["page_content"]
        if len(text) > 500:
            text = text[:500] + "…"
        recital_lines.append(f"Recital ({rn}): {text}")

    return (
        f"--- ARTICLE TO ANALYSE ---\n\n{article_block}\n\n"
        f"--- CANDIDATE RECITALS ({len(candidate_recital_nums)}) ---\n\n"
        + "\n\n".join(recital_lines)
        + f"\n\nIdentify which of these {len(candidate_recital_nums)} recitals provide "
          f"the legislative rationale specifically for Article {article_num}."
    )


# Show prompt size for Article 5
cands5 = [rn for rn, _ in get_candidate_recitals(5, top_k=TOP_K_CANDIDATES)]
msg5   = build_user_message(5, cands5)
print(f"Prompt for Article 5:")
print(f"  Candidates   : {cands5}")
print(f"  Prompt chars : {len(msg5)}  (~{len(msg5)//4} tokens)")
print()
print("First 500 chars:")
print(msg5[:500])

Prompt for Article 5:
  Candidates   : [40, 41, 176, 96, 53, 93, 65, 59, 124, 72, 64, 77, 29, 165, 46, 131, 22, 35, 76, 134, 49, 63, 148, 19, 58, 9, 161, 61, 67, 12]
  Prompt chars : 19120  (~4780 tokens)

First 500 chars:
--- ARTICLE TO ANALYSE ---

ARTICLE 5: Prohibited AI practices
Chapter: CHAPTER II — CHAPTERII
Section:  

1. The following AI practices shall be prohibited:
(a) theplacingonthemarket,theputtingintoserviceortheuseofanAIsystemthatdeployssubliminaltechniquesbeyond
a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of
materiallydistortingthebehaviourofapersonoragroupofpersonsbyappreciablyimpairingtheirabilitytomakean
informeddecision,the


In [15]:
def call_llm(article_num: int, candidate_recital_nums: list,
             verbose: bool = False) -> dict | None:
    """
    Call the LLM for one article.
    Returns {article_num, linked_recitals} or None on failure.
    Always pins explicit metadata refs into the result.
    """
    user_msg = build_user_message(article_num, candidate_recital_nums)

    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            response = llm.invoke([
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=user_msg),
            ])
            raw = response.content.strip()

            if verbose:
                print(f"  Raw response ({len(raw)} chars):")
                print(raw[:600])
                print()

            # Strip markdown fences if model adds them
            raw = re.sub(r"^```json\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"^```\s*",     "", raw, flags=re.MULTILINE)
            raw = raw.strip()

            parsed = json.loads(raw)
            assert "linked_recitals" in parsed
            assert isinstance(parsed["linked_recitals"], list)

            # Clean: deduplicate, keep only real recital nums
            seen, cleaned = set(), []
            for item in parsed["linked_recitals"]:
                rn = int(item["recital_num"])
                if rn in recital_index and rn not in seen:
                    cleaned.append({"recital_num": rn, "reason": item.get("reason", "")})
                    seen.add(rn)

            # Pin explicit refs — belt-and-suspenders
            for expl_rn in sorted(explicit_art_to_recitals.get(article_num, [])):
                if expl_rn not in seen and expl_rn in recital_index:
                    cleaned.append({
                        "recital_num": expl_rn,
                        "reason": "Explicit article reference in recital metadata (guaranteed link)",
                    })

            return {"article_num": article_num, "linked_recitals": cleaned}

        except json.JSONDecodeError as e:
            print(f"  [Attempt {attempt}] JSON parse error: {e}")
            print(f"  Offending text snippet: {raw[:200]}")
        except Exception as e:
            print(f"  [Attempt {attempt}] Error: {type(e).__name__}: {e}")

        if attempt < RETRY_ATTEMPTS:
            time.sleep(RETRY_DELAY)

    print(f"  [FAILED] Article {article_num} exhausted {RETRY_ATTEMPTS} attempts")
    return None


print("call_llm() defined ✓")

call_llm() defined ✓


---
## 8. Single Article Test — Article 5 (Gold Standard)

Article 5: Prohibited AI practices — subliminal manipulation, social scoring,
real-time biometric ID, exploitation of vulnerabilities.
Recitals 28–45 are the known legislative intent block. Let's see what the LLM finds.

In [16]:
print("Article 5: Prohibited AI practices")
print("=" * 60)
print()

cands5   = [rn for rn, _ in get_candidate_recitals(5, top_k=TOP_K_CANDIDATES)]
result5  = call_llm(5, cands5, verbose=True)

if result5:
    linked5   = sorted(result5["linked_recitals"], key=lambda x: x["recital_num"])
    llm_nums5 = [r["recital_num"] for r in linked5]
    print(f"LLM linked {len(linked5)} recitals: {llm_nums5}")
    print()

    if naive_map:
        naive5     = naive_map.get(5, [])
        only_naive = sorted(set(naive5) - set(llm_nums5))
        only_llm   = sorted(set(llm_nums5) - set(naive5))
        print(f"Naive had  : {len(naive5)} → {naive5[:10]}")
        print(f"LLM found  : {len(llm_nums5)}")
        print(f"  Naive-only (likely false positives) : {only_naive}")
        print(f"  LLM-only   (naive missed these)     : {only_llm}")
        print()

    print("LLM reasoning:")
    for r in linked5:
        print(f"  Recital ({r['recital_num']:3d}): {r['reason'][:100]}")

Article 5: Prohibited AI practices

  Raw response (360 chars):
{
  "article_num": 5,
  "linked_recitals": [
    {
      "recital_num": 29,
      "reason": "It explains the specific harms of AI-enabled manipulative techniques, directly justifying the prohibitions in points (a) and (b) against subliminal, manipulative, or deceptive techniques that materially distort human behaviour and cause significant harm."
    }
  ]
}

LLM linked 4 recitals: [29, 40, 41, 176]

Naive had  : 12 → [27, 28, 29, 31, 33, 40, 41, 45, 67, 157]
LLM found  : 4
  Naive-only (likely false positives) : [27, 28, 31, 33, 45, 67, 157, 174]
  LLM-only   (naive missed these)     : []

LLM reasoning:
  Recital ( 29): It explains the specific harms of AI-enabled manipulative techniques, directly justifying the prohib
  Recital ( 40): Explicit article reference in recital metadata (guaranteed link)
  Recital ( 41): Explicit article reference in recital metadata (guaranteed link)
  Recital (176): Explicit article referen

---
## 9. Test on Articles 16 and 53

In [17]:
print("Article 16: Obligations of providers of high-risk AI systems")
print("=" * 62)

cands16  = [rn for rn, _ in get_candidate_recitals(16, top_k=TOP_K_CANDIDATES)]
result16 = call_llm(16, cands16, verbose=False)

if result16:
    linked16 = sorted(result16["linked_recitals"], key=lambda x: x["recital_num"])
    nums16   = [r["recital_num"] for r in linked16]
    print(f"LLM linked {len(linked16)} recitals: {nums16}")
    if naive_map:
        naive16 = naive_map.get(16, [])
        print(f"Naive had  : {len(naive16)} recitals: {naive16[:10]}")
        print(f"Reduced by : {len(set(naive16)-set(nums16))} (false positives removed)")
    print()
    for r in linked16:
        text = recital_index[r["recital_num"]]["page_content"][:100].replace("\n", " ")
        print(f"  Recital ({r['recital_num']:3d}): {r['reason'][:90]}")
        print(f"             → '{text}…'")
        print()

Article 16: Obligations of providers of high-risk AI systems
LLM linked 6 recitals: [4, 40, 84, 124, 131, 136]
Naive had  : 95 recitals: [3, 4, 7, 9, 10, 14, 20, 21, 22, 26]
Reduced by : 89 (false positives removed)

  Recital (  4): Explicit article reference in recital metadata (guaranteed link)
             → 'Positionof theEuropeanParliamentof13March2024(notyetpublishedintheOfficialJournal)anddecisionof theC…'

  Recital ( 40): Explicit article reference in recital metadata (guaranteed link)
             → 'InaccordancewithArticle6aofProtocolNo21onthepositionoftheUnitedKingdomandIrelandinrespectofthe areao…'

  Recital ( 84): Clarifies the conditions under which distributors, importers, deployers or other third par
             → 'To ensure legalcertainty, it is necessary toclarify that, under certain specific conditions, anydist…'

  Recital (124): Explains the rationale for integrating conformity assessment with existing Union harmonisa
             → 'Itisappropriatethat,inorder

In [18]:
print("Article 53: Obligations for providers of general-purpose AI models (GPAI)")
print("=" * 70)
print("Added late in the legislative process — naive map likely struggled most here.")
print()

cands53  = [rn for rn, _ in get_candidate_recitals(53, top_k=TOP_K_CANDIDATES)]
result53 = call_llm(53, cands53, verbose=False)

if result53:
    linked53 = sorted(result53["linked_recitals"], key=lambda x: x["recital_num"])
    nums53   = [r["recital_num"] for r in linked53]
    print(f"LLM linked {len(linked53)} recitals: {nums53}")
    if naive_map:
        print(f"Naive had  : {naive_map.get(53, [])[:10]}")
    print()
    for r in linked53:
        print(f"  Recital ({r['recital_num']:3d}): {r['reason'][:110]}")

Article 53: Obligations for providers of general-purpose AI models (GPAI)
Added late in the legislative process — naive map likely struggled most here.

LLM linked 6 recitals: [21, 22, 106, 109, 117, 148]
Naive had  : [10, 14, 20, 72, 76, 81, 84, 85, 86, 89]

  Recital ( 21): Clarifies the scope of Article 53 by stating that its rules apply to providers of AI systems (including genera
  Recital ( 22): Supports the extraterritorial application of Article 53 by explaining that the Regulation's rules apply to AI 
  Recital (106): Justifies the specific obligation in Article 53(1)(c) by explaining that providers of general-purpose AI model
  Recital (109): Provides the legislative rationale for the proportionality of obligations and the open-source exception in Art
  Recital (117): Explains the purpose of Article 53(4) by establishing codes of practice as a central tool for providers of gen
  Recital (148): Contextualises the governance framework referenced in Article 53(3) and the role of

---
## 10. Batch All 113 Articles

Progress saved every 10 articles to `_arm_intermediate.json`.  
Re-running this cell resumes from where it left off.

In [19]:
# Resume support
completed = {}
if INTERMEDIATE.exists():
    with open(INTERMEDIATE) as f:
        completed = {int(k): v for k, v in json.load(f).items()}
    print(f"Resuming: {len(completed)} articles already done")
else:
    print("Starting fresh")

remaining = [n for n in article_nums_sorted if n not in completed]
print(f"Articles remaining: {len(remaining)} / {len(article_nums_sorted)}")
print()

failed = []

for art_num in tqdm(remaining, desc="Mapping articles"):
    cands  = [rn for rn, _ in get_candidate_recitals(art_num, top_k=TOP_K_CANDIDATES)]
    result = call_llm(art_num, cands, verbose=False)

    if result is not None:
        completed[art_num] = result
    else:
        failed.append(art_num)
        completed[art_num] = {"article_num": art_num, "linked_recitals": [], "_failed": True}

    if len(completed) % 10 == 0:
        with open(INTERMEDIATE, "w") as f:
            json.dump(completed, f, indent=2)

    time.sleep(REQUEST_DELAY)

with open(INTERMEDIATE, "w") as f:
    json.dump(completed, f, indent=2)

print(f"\nBatch complete.")
print(f"  Succeeded : {len(completed) - len(failed)}")
print(f"  Failed    : {len(failed)} → {failed}")

Starting fresh
Articles remaining: 113 / 113



Mapping articles: 100%|████████████████████████████████| 113/113 [1:29:53<00:00, 47.73s/it]


Batch complete.
  Succeeded : 113
  Failed    : 0 → []


In [20]:
# Retry failed articles with a narrower candidate window
# Long prompts are the most common failure cause

retry_targets = [n for n, r in completed.items() if r.get("_failed")]

if retry_targets:
    print(f"Retrying {len(retry_targets)} failed articles with top-15 candidates …")
    for art_num in retry_targets:
        cands  = [rn for rn, _ in get_candidate_recitals(art_num, top_k=15)]
        result = call_llm(art_num, cands, verbose=True)
        if result:
            completed[art_num] = result
            print(f"  ✓ Article {art_num} recovered")
        time.sleep(RETRY_DELAY)

    with open(INTERMEDIATE, "w") as f:
        json.dump(completed, f, indent=2)
    print("Retries done ✓")
else:
    print("No failed articles ✓")

No failed articles ✓


---
## 11. Build Final Map & Compare vs Naive

In [21]:
ARTICLE_RECITAL_MAP_LLM = {}   # int → list[int]  (the drop-in replacement map)
ARTICLE_RECITAL_REASONS = {}   # int → list[{recital_num, reason}]  (audit trail)

for art_num in article_nums_sorted:
    result = completed.get(art_num)

    if result and not result.get("_failed"):
        linked = result["linked_recitals"]
        ARTICLE_RECITAL_MAP_LLM[art_num] = sorted({r["recital_num"] for r in linked})
        ARTICLE_RECITAL_REASONS[art_num] = sorted(linked, key=lambda x: x["recital_num"])
    else:
        # Fallback: explicit refs only
        expl = sorted(explicit_art_to_recitals.get(art_num, []))
        ARTICLE_RECITAL_MAP_LLM[art_num] = expl
        ARTICLE_RECITAL_REASONS[art_num] = [
            {"recital_num": rn, "reason": "Explicit ref (LLM call failed — fallback)"}
            for rn in expl
        ]

covered   = sum(1 for v in ARTICLE_RECITAL_MAP_LLM.values() if v)
total_ass = sum(len(v) for v in ARTICLE_RECITAL_MAP_LLM.values())

print("LLM Map built ✓")
print(f"  Articles covered          : {covered} / {len(article_nums_sorted)}")
print(f"  Total assignments         : {total_ass}")
print(f"  Avg recitals per article  : {total_ass/max(covered,1):.1f}")
print()
print(f"  {'Art':>4}  {'Title':<48}  Recitals")
print("  " + "-" * 78)
for n in [5, 6, 9, 10, 16, 26, 43, 53, 99]:
    recs   = ARTICLE_RECITAL_MAP_LLM.get(n, [])[:6]
    suffix = "…" if len(ARTICLE_RECITAL_MAP_LLM.get(n,[])) > 6 else ""
    title  = full_articles[n]["article_title"]
    print(f"  {n:4d}  {title[:48]:<48}  {recs}{suffix}")

LLM Map built ✓
  Articles covered          : 103 / 113
  Total assignments         : 358
  Avg recitals per article  : 3.5

   Art  Title                                             Recitals
  ------------------------------------------------------------------------------
     5  Prohibited AI practices                           [19, 29, 35, 40, 41, 59]…
     6  Classification rules for high-risk AI systems     [6, 49, 53, 55, 63, 131]
     9  Risk management system                            [64, 65]
    10  Data and data governance                          [38, 39, 67, 94, 122, 140]
    16  Obligations of providers of high-risk AI systems  [4, 9, 40, 46, 49, 64]…
    26  Obligations of deployers of high-risk AI systems  [72, 91, 93, 96, 158]
    43  Conformity assessment                             [9, 46, 77, 121]
    53  Obligations for providers of general-purpose AI   [106, 109, 112, 117, 148]
    99  Penalties                                         [156]


In [22]:
if not naive_map:
    print("No naive map to compare against.")
else:
    naive_total   = sum(len(v) for v in naive_map.values())
    naive_covered = sum(1 for v in naive_map.values() if v)

    print("=" * 70)
    print("COMPARISON: LLM map vs Naive 3-signal map")
    print("=" * 70)
    print(f"\n  Coverage (articles with ≥1 recital):")
    print(f"    Naive : {naive_covered} / {len(full_articles)}")
    print(f"    LLM   : {covered}  / {len(full_articles)}")
    print(f"\n  Total assignments (lower = more selective):")
    print(f"    Naive : {naive_total}  (avg {naive_total/max(naive_covered,1):.1f}/article)")
    print(f"    LLM   : {total_ass}   (avg {total_ass/max(covered,1):.1f}/article)")

    big_drops, big_gains = [], []
    for n in article_nums_sorted:
        ns = set(naive_map.get(n, []))
        ls = set(ARTICLE_RECITAL_MAP_LLM.get(n, []))
        delta = len(ls) - len(ns)
        if delta <= -5:
            big_drops.append((n, full_articles[n]["article_title"], ns, ls))
        elif delta >= 3:
            big_gains.append((n, full_articles[n]["article_title"], ns, ls))

    print(f"\n  Articles where LLM removed ≥5 (naive over-assigned):")
    for n, title, ns, ls in big_drops[:8]:
        print(f"    Art {n:3d} ({title[:35]:<35}) -{len(ns)-len(ls):2d}  removed={sorted(ns-ls)[:5]}")

    print(f"\n  Articles where LLM added ≥3 (naive missed):")
    for n, title, ns, ls in big_gains[:8]:
        print(f"    Art {n:3d} ({title[:35]:<35}) +{len(ls)-len(ns):2d}  added={sorted(ls-ns)[:5]}")

COMPARISON: LLM map vs Naive 3-signal map

  Coverage (articles with ≥1 recital):
    Naive : 108 / 113
    LLM   : 103  / 113

  Total assignments (lower = more selective):
    Naive : 1428  (avg 13.2/article)
    LLM   : 358   (avg 3.5/article)

  Articles where LLM removed ≥5 (naive over-assigned):
    Art   5 (Prohibited AI practices            ) - 5  removed=[27, 28, 31, 33, 45]
    Art   6 (Classification rules for high-risk ) -76  removed=[3, 7, 8, 9, 14]
    Art   8 (Compliance with the requirements   ) -11  removed=[9, 10, 46, 67, 69]
    Art   9 (Risk management system             ) -15  removed=[55, 57, 60, 66, 67]
    Art  13 (Transparency and provision of infor) -16  removed=[9, 10, 22, 33, 66]
    Art  16 (Obligations of providers of high-ri) -80  removed=[3, 7, 10, 14, 20]
    Art  17 (Quality management system          ) -21  removed=[27, 31, 55, 56, 57]
    Art  21 (Cooperation with competent authorit) -18  removed=[10, 13, 22, 60, 68]

  Articles where LLM added ≥3 (n

In [23]:
# Read the removed recitals for the worst-dropped article
# to see concretely why naive over-assigned them

if naive_map:
    worst_n = max(
        article_nums_sorted,
        key=lambda n: len(set(naive_map.get(n,[])) - set(ARTICLE_RECITAL_MAP_LLM.get(n,[])))
    )
    removed = sorted(set(naive_map.get(worst_n,[])) - set(ARTICLE_RECITAL_MAP_LLM.get(worst_n,[])))
    title   = full_articles[worst_n]["article_title"]

    print(f"Article with most naive false-positives removed: {worst_n} — {title}")
    print(f"Removed recitals: {removed}")
    print()
    print("First 3 removed recitals — should look like generic boilerplate:")
    for rn in removed[:3]:
        text = recital_index.get(rn, {}).get("page_content", "")[:280].replace("\n", " ")
        print(f"  Recital ({rn:3d}): {text}…")
        print()

Article with most naive false-positives removed: 16 — Obligations of providers of high-risk AI systems
Removed recitals: [3, 7, 10, 14, 20, 21, 22, 26, 50, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 66, 68, 69, 70, 71, 72, 73, 74, 75, 78, 80, 81, 82, 83, 85, 86, 87, 88, 90, 91, 92, 96, 97, 101, 108, 109, 112, 114, 116, 117, 118, 120, 123, 125, 128, 129, 130, 132, 133, 137, 139, 141, 143, 145, 146, 155, 156, 157, 158, 159, 160, 161, 162, 165, 166, 171, 173, 174, 177, 178, 179]

First 3 removed recitals — should look like generic boilerplate:
  Recital (  3): AIsystemscanbeeasilydeployedinalargevarietyofsectorsoftheeconomyandmanypartsofsociety,including across borders, and can easily circulate throughout the Union. Certain Member States have already explored the adoptionof national rules toensure that AI is trustworthy and safeand is …

  Recital (  7): In order to ensure a consistent and high level of protection of public interests as regards health, safety and fundamentalrights,common

---
## 12. Quality Audit — 10 Articles

In [24]:
AUDIT_ARTICLES = [3, 5, 9, 10, 16, 26, 43, 53, 71, 99]

for art_num in AUDIT_ARTICLES:
    art    = full_articles[art_num]
    linked = ARTICLE_RECITAL_REASONS.get(art_num, [])
    nums   = [r["recital_num"] for r in linked]

    print()
    print("=" * 70)
    print(f"Article {art_num}: {art['article_title']}")
    print(f"  {art['chapter_num']} — {art['chapter_title']}")
    print(f"  Chars: {art['char_len']}  Chunks: {art['num_chunks']}  Recitals: {nums}")
    print()

    if not linked:
        print("  [No recitals assigned]")
        continue

    for r in sorted(linked, key=lambda x: x["recital_num"]):
        rn   = r["recital_num"]
        text = recital_index[rn]["page_content"][:200].replace("\n", " ")
        print(f"  Recital ({rn:3d}): {text}…")
        print(f"    └─ LLM: {r['reason'][:105]}")
        print()


Article 3: Definitions
  CHAPTER I — CHAPTERI
  Chars: 17793  Chunks: 13  Recitals: [53, 74, 84, 100, 128]

  Recital ( 53): It is also important to clarify that there may be specific cases in whichAI systems referred to in pre-defined areas specifiedinthisRegulationdonotleadtoasignificantriskofharmtothelegalinterestsprotec…
    └─ LLM: Explicit article reference in recital metadata (guaranteed link)

  Recital ( 74): High-risk AI systems should perform consistently throughout their lifecycle and meet an appropriate level of accuracy, robustness and cybersecurity, in light of their intended purpose and in accordanc…
    └─ LLM: Specifies that performance metrics should be declared in instructions for use, providing rationale for th

  Recital ( 84): To ensure legalcertainty, it is necessary toclarify that, under certain specific conditions, anydistributor, importer, deployerorotherthird-partyshouldbeconsideredtobeaproviderofahigh-riskAIsystemandt…
    └─ LLM: Explains when a distribut

---
## 13. Strict Second Pass for Uncovered Articles

Distinguish *true no-recital articles* (highly procedural provisions) from *missed links*.
We re-run with top-40 candidates and an explicit instruction to return empty if nothing applies.

In [25]:
uncovered = [
    (n, full_articles[n]["article_title"])
    for n in article_nums_sorted
    if not ARTICLE_RECITAL_MAP_LLM.get(n)
]

print(f"Articles with no recitals after main batch: {len(uncovered)}")
for n, title in uncovered:
    print(f"  Art {n:3d}: {title[:58]}")
    print(f"    → '{full_articles[n]['full_text'][:180].replace(chr(10),' ')}…'")

print()

Articles with no recitals after main batch: 10
  Art  31: Requirements relating to notified bodies
    → '1. A notified body shall be established under the national law of a Member State and shall have legal personality. 2. Notified bodies shall satisfy the organisational, quality mana…'
  Art  33: Subsidiaries of notified bodies and subcontracting
    → '1. Where a notified body subcontracts specific tasks connected with the conformity assessment or has recourse to a subsidiary, it shall ensure that the subcontractor or the subsidi…'
  Art  35: Identification numbers and lists of notified bodies
    → '1. TheCommissionshallassignasingleidentificationnumbertoeachnotifiedbody,evenwhereabodyisnotifiedunder more than one Union act. 2. The Commission shall make publicly available the …'
  Art  36: Changes to notifications
    → '1. The notifying authority shall notify the Commission and the other Member States of any relevant changes to the notification of a notified body via the electroni

In [26]:
STRICT_SYSTEM = (
    SYSTEM_PROMPT
    + "\n\nIMPORTANT: If NONE of the candidates genuinely explain this article's specific "
      "purpose, return: {\"article_num\": N, \"linked_recitals\": []}"
)

if uncovered:
    print(f"Running strict second pass on {len(uncovered)} uncovered articles (top-40 candidates) …")
    print()
    recovered = 0

    for n, title in uncovered:
        cands    = [rn for rn, _ in get_candidate_recitals(n, top_k=40)]
        user_msg = build_user_message(n, cands)

        for attempt in range(RETRY_ATTEMPTS):
            try:
                response = llm.invoke([
                    SystemMessage(content=STRICT_SYSTEM),
                    HumanMessage(content=user_msg),
                ])
                raw    = response.content.strip()
                raw    = re.sub(r"^```json\s*", "", raw, flags=re.MULTILINE)
                raw    = re.sub(r"^```\s*",     "", raw, flags=re.MULTILINE)
                parsed = json.loads(raw.strip())

                if parsed.get("linked_recitals"):
                    cleaned = [
                        {"recital_num": int(r["recital_num"]), "reason": r.get("reason","")}
                        for r in parsed["linked_recitals"]
                        if int(r["recital_num"]) in recital_index
                    ]
                    nums = [r["recital_num"] for r in cleaned]
                    ARTICLE_RECITAL_MAP_LLM[n] = sorted(nums)
                    ARTICLE_RECITAL_REASONS[n] = sorted(cleaned, key=lambda x: x["recital_num"])
                    print(f"  ✓ Art {n:3d} ({title[:35]:<35}) recovered → {nums}")
                    recovered += 1
                else:
                    print(f"  ─ Art {n:3d} ({title[:35]:<35}) confirmed: no recitals")
                break

            except Exception as e:
                print(f"  Art {n} attempt {attempt+1}: {e}")
                time.sleep(RETRY_DELAY)

        time.sleep(REQUEST_DELAY)

    print(f"\nStrict pass: {recovered} recovered, "
          f"{len(uncovered)-recovered} confirmed as no-recital")
else:
    print("No uncovered articles — skipping strict pass ✓")

Running strict second pass on 10 uncovered articles (top-40 candidates) …

  ✓ Art  31 (Requirements relating to notified b) recovered → [9, 46, 64, 156]
  ─ Art  33 (Subsidiaries of notified bodies and) confirmed: no recitals
  ─ Art  35 (Identification numbers and lists of) confirmed: no recitals
  ─ Art  36 (Changes to notifications           ) confirmed: no recitals
  ─ Art  39 (Conformity assessment bodies of thi) confirmed: no recitals
  ─ Art  87 (Reporting of infringements and prot) confirmed: no recitals
  ─ Art  97 (Exercise of the delegation         ) confirmed: no recitals
  ─ Art 100 (Administrative fines on Union insti) confirmed: no recitals
  ─ Art 110 (Amendment to Directive (EU) 2020/18) confirmed: no recitals
  ✓ Art 113 (Entry into force and application   ) recovered → [179]

Strict pass: 2 recovered, 8 confirmed as no-recital


---
## 14. Save Final Map

In [27]:
# Clean map
with open(OUTPUT_MAP, "w") as f:
    json.dump(ARTICLE_RECITAL_MAP_LLM, f, indent=2)
print(f"Saved: {OUTPUT_MAP}")

# Audit trail
with open(OUTPUT_REASONS, "w") as f:
    json.dump(
        {
            str(n): {
                "article_title"  : full_articles[n]["article_title"],
                "linked_recitals": reasons,
            }
            for n, reasons in sorted(ARTICLE_RECITAL_REASONS.items())
        },
        f, indent=2, ensure_ascii=False
    )
print(f"Saved: {OUTPUT_REASONS}")

# Final stats
covered_final   = sum(1 for v in ARTICLE_RECITAL_MAP_LLM.values() if v)
total_final     = sum(len(v) for v in ARTICLE_RECITAL_MAP_LLM.values())
uncovered_final = [n for n in article_nums_sorted if not ARTICLE_RECITAL_MAP_LLM.get(n)]

print()
print("─" * 55)
print("FINAL MAP STATS")
print("─" * 55)
print(f"  Articles covered          : {covered_final} / {len(article_nums_sorted)}")
print(f"  Total assignments         : {total_final}")
print(f"  Avg recitals per article  : {total_final/max(covered_final,1):.1f}")
print(f"  Articles with no recitals : {len(uncovered_final)} → {uncovered_final}")
print("─" * 55)

Saved: chunker_output/article_recital_map_llm.json
Saved: chunker_output/article_recital_reasons_llm.json

───────────────────────────────────────────────────────
FINAL MAP STATS
───────────────────────────────────────────────────────
  Articles covered          : 105 / 113
  Total assignments         : 363
  Avg recitals per article  : 3.5
  Articles with no recitals : 8 → [33, 35, 36, 39, 87, 97, 100, 110]
───────────────────────────────────────────────────────


In [28]:
print("""
HOW TO USE IN eu_ai_act_retrieval.ipynb / eu_ai_act_agent.ipynb
================================================================

Replace:

    with open(DATA_DIR / "article_recital_map.json") as f:
        arm_raw = json.load(f)
    ARTICLE_RECITAL_MAP = {int(k): v for k, v in arm_raw.items()}

With:

    with open(DATA_DIR / "article_recital_map_llm.json") as f:
        arm_raw = json.load(f)
    ARTICLE_RECITAL_MAP = {int(k): v for k, v in arm_raw.items()}

That's it. The format is identical — int keys, list[int] values.
get_recitals_for_article() and all downstream code work unchanged.

To debug a bad answer: open article_recital_reasons_llm.json,
find the article number, and read the LLM's reason for each assignment.
""")

# Verify the saved file loads correctly
with open(OUTPUT_MAP) as f:
    verify = {int(k): v for k, v in json.load(f).items()}
print(f"Verification load: {len(verify)} articles ✓")
print()

# Quick function test
def get_recitals_for_article(article_num: int, top_k: int = 4) -> list:
    nums = verify.get(article_num, [])[:top_k]
    return [
        {"recital_num": n, "text": recital_index[n]["page_content"][:120]}
        for n in nums if n in recital_index
    ]

print("get_recitals_for_article(5, top_k=4):")
for r in get_recitals_for_article(5, top_k=4):
    print(f"  Recital ({r['recital_num']:3d}): {r['text'].replace(chr(10),' ')}…")

print()
print("get_recitals_for_article(16, top_k=3):")
for r in get_recitals_for_article(16, top_k=3):
    print(f"  Recital ({r['recital_num']:3d}): {r['text'].replace(chr(10),' ')}…")


HOW TO USE IN eu_ai_act_retrieval.ipynb / eu_ai_act_agent.ipynb

Replace:

    with open(DATA_DIR / "article_recital_map.json") as f:
        arm_raw = json.load(f)
    ARTICLE_RECITAL_MAP = {int(k): v for k, v in arm_raw.items()}

With:

    with open(DATA_DIR / "article_recital_map_llm.json") as f:
        arm_raw = json.load(f)
    ARTICLE_RECITAL_MAP = {int(k): v for k, v in arm_raw.items()}

That's it. The format is identical — int keys, list[int] values.
get_recitals_for_article() and all downstream code work unchanged.

To debug a bad answer: open article_recital_reasons_llm.json,
find the article number, and read the LLM's reason for each assignment.

Verification load: 113 articles ✓

get_recitals_for_article(5, top_k=4):
  Recital ( 19): ForthepurposesofthisRegulationthenotionof‘publiclyaccessiblespace’shouldbeunderstoodasreferringtoany physical space that…
  Recital ( 29): AI-enabled manipulative techniques can be used to persuade persons to engage in unwanted behaviours, o

---
## Summary

| Step | What we did | Why |
|---|---|---|
| **Reconstruct** | Merged multi-chunk articles by `article_num + chunk_index` | Articles span multiple pages — LLM needs full context |
| **Phantom guard** | Filtered `explicit_art_to_recitals` to `known_article_nums` | Chunker regex catches `Article 261` from other EU regs — crashes without guard |
| **Embed** | `qwen/qwen3-embedding-8b` via OpenRouter + LangChain | Understands formal legal text, bridges rationale↔obligation language gap |
| **Guaranteed layer** | Explicit refs pinned at score=2.0 regardless of embedding rank | 100% recall on metadata-confirmed links by construction |
| **K-sweep** | Measured embedding-only recall at K=10…100 | Empirical basis for TOP_K_CANDIDATES choice |
| **LLM judge** | `step-3.5-flash:free` at temp=0, JSON output | Judges legislative intent, not vocabulary overlap |
| **Explicit pin** | Post-process: always include explicit refs in LLM output | Never lose guaranteed links even if LLM misses them |
| **Resume** | Intermediate save every 10 articles | Safe against API interruptions |
| **Retry** | Failed articles retried with top-15 | Shorter prompts fix token-limit failures |
| **Strict pass** | Uncovered articles re-run with top-40 + no-link instruction | True no-recital articles vs genuine misses |
| **Audit trail** | `article_recital_reasons_llm.json` with one reason per link | Every assignment is human-readable and debuggable |
